# Embedding model screening on a corpus sample (Colab T4)

Compares three candidate embedding models with **BM25+** and **BGE-base** on the same
~20,000-document sample, alone and merged with BM25+ (RRF, BM25 weight 0.7). The most
promising model is then worth a full-corpus run (on an L4).

- **Candidates:** `gte-modernbert-base` (149M), `arctic-embed-l-v2.0` (568M) and
  `qwen3-embedding-0.6b` (596M). All are open (Apache-2.0) and pinned to exact versions.
- **Sample:** every reference document for the 300 development questions, the documents
  BM25+ and BGE-base ranked in their top 20 for those questions on the full corpus (the
  look-alikes), and 10,000 random documents. Calibration and final questions are not used.
- **Runtime → Change runtime type → T4 GPU.** High-RAM is not needed. Expect about 2 hours:
  roughly 10 min each for BGE-base and GTE, and 45–60 min each for Arctic and Qwen3.
  Each model's embeddings are saved to Drive, so after a disconnect rerun all cells to resume.

Scores on a sample are higher than on the full corpus. Compare rows with each other. The
sample includes BM25+'s and BGE-base's own top competitors but not the new models', so a
candidate should beat BGE-base clearly before it gets a full-corpus run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Drive and project code

In [ ]:
import json
import os
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"
CORPUS = f"{WORK}/documents.parquet"
QUESTIONS = f"{WORK}/questions.jsonl"
SAMPLE = f"{WORK}/sample"  # Sample documents, its BM25+ index and each model's embeddings.

BRANCH = "hira/dev"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} https://github.com/hira-mian/Agentic-Enterprise-RAG-System.git /content/project
%cd /content/project
!git log -1 --format='%h %s'
!pip install -q -r requirements.txt

## 2. Build the sample (under a minute)

Uses the saved full-corpus BM25+ and BGE-base development results in the repository. The
document list fingerprint should match the one shown, so everyone screens the same sample.

In [ ]:
EXPECTED = "6e6eecccd67afc7c02f69151e95aa4698a9ed9233a2b3be9fbcf4e4f7ce855a2"
if not os.path.exists(f"{SAMPLE}/manifest.json"):
    !python -m src.data.sample --corpus {CORPUS} --questions {QUESTIONS} --output {SAMPLE}
sample = json.load(open(f"{SAMPLE}/manifest.json"))
print({k: sample[k] for k in ("documents", "reference", "lookalikes", "random")})
if sample["doc_ids_sha256"] != EXPECTED:
    print("Note: this sample differs from the reference one (different corpus copy?). "
          "Results are still valid among themselves; mention it when sharing them.")
if not os.path.exists(f"{SAMPLE}/bm25/manifest.json"):
    !python -m src.retrieval.corpus_bm25 --corpus {SAMPLE}/documents.parquet --output {SAMPLE}/bm25

## 3. Embed the sample with each model

Same settings as the full-corpus BGE-base run: 500-token passages with 50-token overlap and
the title on every passage, in fp16. Finished models are skipped on rerun. If a model stops
with "non-finite embeddings", rerun that model with `--precision fp32` (slower).

In [ ]:
MODELS = {  # Model label -> GPU batch size for a T4.
    "bge-base": 128,
    "gte-modernbert-base": 128,
    "arctic-embed-l-v2.0": 32,
    "qwen3-embedding-0.6b": 32,
}
for label, batch in MODELS.items():
    print(f"=== {label}", flush=True)
    !python -m src.retrieval.corpus_dense --corpus {SAMPLE}/documents.parquet --output {SAMPLE}/{label} \
        --model {label} --device cuda --batch-size {batch} --shard-docs 5000 --title-every-passage

## 4. Compare on the development questions

Columns: Recall@10, nDCG@10, how many of the questions BM25+ misses (nothing correct in its
top 10) the run finds, and questions better / worse than BM25+ on Recall@10.

In [ ]:
DENSE = " ".join(f"{label}={SAMPLE}/{label}" for label in MODELS)
!python -m src.evaluation.screen_models --sample {SAMPLE} --dense {DENSE} --device cuda \
    --questions {QUESTIONS} --output-dir evaluation/results/screening

## 5. Save the results

Copies the result folders to Drive (`rag-dense/results/screening`) and downloads
`screening_results.zip`. Upload the zip to add the results to the repository.

In [ ]:
!mkdir -p {WORK}/results && cp -r evaluation/results/screening {WORK}/results/
!cd evaluation/results && zip -qr /content/screening_results.zip screening
from google.colab import files
files.download("/content/screening_results.zip")